In [1]:
import sagemaker

# Initialize session and IAM role
sess = sagemaker.Session()
role = sagemaker.get_execution_role()

# S3 Bucket for storing lab assets
bucket = "ed-ai-training-eyob-2026"

print("✅ SageMaker Session Active")
print(f"├── Region: {sess.boto_region_name}")
print(f"├── Role: {role}")
print(f"└── Bucket: {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
✅ SageMaker Session Active
├── Region: eu-north-1
├── Role: arn:aws:iam::343147895485:role/service-role/AmazonSageMaker-ExecutionRole-20260721T132003
└── Bucket: ed-ai-training-eyob-2026


In [2]:
%%writefile train.py
import argparse
import os
import joblib
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

if __name__ == "__main__":
    # 1. Read input dataset mounted inside the Docker container
    train_dir = os.environ.get("SM_CHANNEL_TRAIN", "./local_data")
    file_path = os.path.join(train_dir, "soldiers_fitness.csv")
    
    print(f"Loading dataset from: {file_path}")
    df = pd.read_csv(file_path)
    
    # 2. Prepare features and target label
    X = df[["age", "pushups_per_min", "run_5km_minutes"]]
    y = df["fitness_level"]
    
    # 3. Train Decision Tree classifier
    print("Training DecisionTreeClassifier model...")
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X, y)
    
    # 4. Save model artifact into SageMaker model directory
    model_dir = os.environ.get("SM_MODEL_DIR", "./")
    output_path = os.path.join(model_dir, "model.joblib")
    
    joblib.dump(model, output_path)
    print(f"Training complete. Model saved to: {output_path}")

Writing train.py


In [4]:
import os
import boto3
import pandas as pd
from sagemaker.sklearn.estimator import SKLearn

# A. Prepare local directory
os.makedirs("local_data", exist_ok=True)
local_data_path = "local_data/soldiers_fitness.csv"

# B. Raw URL for your specific GitHub repository file
raw_github_url = "https://raw.githubusercontent.com/eyobn/ED-AI-Training/main/sample_dataset"

print(f"📥 Fetching dataset from: {raw_github_url}")
try:
    df = pd.read_csv(raw_github_url)
    df.to_csv(local_data_path, index=False)
    print(f"✅ Successfully downloaded dataset to '{local_data_path}'!")
except Exception as e:
    print(f"❌ Failed to download dataset: {e}")

# C. Define Local SKLearn Estimator (bypasses AWS cloud quotas)
est = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="local",  # Runs container locally on notebook instance
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3"
)

# D. Launch local training job
print("🚀 Starting local SageMaker training container...")
est.fit({"train": "file://local_data"})

📥 Fetching dataset from: https://raw.githubusercontent.com/eyobn/ED-AI-Training/main/sample_dataset


INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


✅ Successfully downloaded dataset to 'local_data/soldiers_fitness.csv'!


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


🚀 Starting local SageMaker training container...


INFO:sagemaker:Creating training-job with name: sagemaker-scikit-learn-2026-07-21-10-34-29-246
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' is not installed. Proceeding to check for 'docker-compose' CLI.
INFO:sagemaker.local.image:'Docker Compose' found using Docker Compose CLI.
INFO:sagemaker.local.local_session:Starting training job
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:sagemaker.local.image:No AWS credentials found in session but credentials from EC2 Metadata Service are available.
INFO:sagemaker.local.

 Container b3viufocyo-algo-1-hgwsz  Creating
 Container b3viufocyo-algo-1-hgwsz  Created
Attaching to b3viufocyo-algo-1-hgwsz
b3viufocyo-algo-1-hgwsz  | /miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
b3viufocyo-algo-1-hgwsz  |   import pkg_resources
b3viufocyo-algo-1-hgwsz  | 2026-07-21 10:34:30,777 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
b3viufocyo-algo-1-hgwsz  | 2026-07-21 10:34:30,782 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
b3viufocyo-algo-1-hgwsz  | 2026-07-21 10:34:30,786 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
b3viufocyo-algo-1-hgwsz  | 2026-07-21 10:34:30,795 sagemaker-trainin

INFO:sagemaker.local.image:===== Job Complete =====


b3viufocyo-algo-1-hgwsz exited with code 0
Aborting on container exit...
 Container b3viufocyo-algo-1-hgwsz  Stopping
 Container b3viufocyo-algo-1-hgwsz  Stopped


In [6]:
# 1. Ensure scikit-learn and joblib are installed in the notebook kernel
!pip install -q scikit-learn joblib

import os
import joblib
import boto3
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# 2. Read dataset directly from local folder
df = pd.read_csv("local_data/soldiers_fitness.csv")
X = df[["age", "pushups_per_min", "run_5km_minutes"]]
y = df["fitness_level"]

# 3. Fit DecisionTree model in active notebook kernel
clf = DecisionTreeClassifier(random_state=42).fit(X, y)

# 4. Save model file directly in notebook workspace
joblib.dump(clf, "model.joblib")
print("✅ Saved 'model.joblib' directly in notebook workspace.")

# 5. Upload model artifact to Amazon S3
s3 = boto3.client("s3")
s3.upload_file("model.joblib", bucket, "models/model.joblib")
print(f"🚀 Successfully uploaded model to s3://{bucket}/models/model.joblib")

✅ Saved 'model.joblib' directly in notebook workspace.
🚀 Successfully uploaded model to s3://ed-ai-training-eyob-2026/models/model.joblib


In [8]:
# Cell 4 — Load local trained model into memory
import os
import joblib

model_file = "model.joblib"

if os.path.exists(model_file):
    predictor = joblib.load(model_file)
    print("✅ Model loaded successfully into active kernel workspace!")
else:
    raise FileNotFoundError(f"❌ '{model_file}' not found. Please re-run Cell 4 from earlier to generate it.")

✅ Model loaded successfully into active kernel workspace!


In [9]:
# Cell 5 — Invoke local predictor with recruit data
import pandas as pd

# Define test sample: Recruit 1 (24 yrs, 48 pushups, 23 min 5k) | Recruit 2 (40 yrs, 15 pushups, 36 min 5k)
recruits = pd.DataFrame(
    [[24, 48, 23], [40, 15, 36]],
    columns=["age", "pushups_per_min", "run_5km_minutes"]
)

# Generate local predictions
predictions = predictor.predict(recruits)

print("\n--------------------------------------------------")
print(f"📥 Test Data Sent:\n{recruits.to_string(index=False)}")
print("--------------------------------------------------")
print(f"🔮 Model Predictions: {predictions.tolist()}")
print("--------------------------------------------------\n")


--------------------------------------------------
📥 Test Data Sent:
 age  pushups_per_min  run_5km_minutes
  24               48               23
  40               15               36
--------------------------------------------------
🔮 Model Predictions: ['high', 'low']
--------------------------------------------------

